---
title: Multi-level caching
---

Phasic uses a four-layer caching system to avoid repeating expensive computations. Each layer targets a different bottleneck in the pipeline from model definition to inference:

| Layer | What is cached | Location | When it helps |
|-------|---------------|----------|---------------|
| **Graph cache** | Fully constructed `Graph` objects | `~/.phasic_cache/graphs/` | Avoids callback-based construction |
| **Parameterised reward-compute cache** | Symbolic elimination output, theta-independent | `~/.phasic_cache/parameterized_reward_compute/<hash>.bin` | Skips the O(n³) Gaussian elimination across processes |
| **Hierarchical SCC composer cache** | Per-SCC parameterised reward-compute artefacts | `~/.phasic_cache/parameterized_reward_compute/scc_<hash>.bin` | Parallel SCC elimination, cross-graph SCC reuse |
| **JAX compilation cache** | JIT-compiled XLA code | `~/.jax_cache/` | Avoids recompilation on restart |

All caches are **persistent** — they survive across Python sessions and restarts. Cache correctness is ensured by SHA-256 content hashing: the same graph structure always produces the same hash, and any structural change automatically invalidates the entry. The parameterised reward-compute and SCC composer caches are **theta-independent** — re-running the same parameterised model with new theta values does not produce new cache entries.

In [ ]:
from phasic import (
    Graph, Property, StateIndexer, set_log_level,
    clear_caches, clear_model_cache,
    cache_info, get_all_cache_stats, print_all_cache_info,
    get_graph_cache_stats, print_graph_cache_info,
    configure,
)
import phasic.cache as cache
import sys
import numpy as np
import time
from vscodenb import set_vscode_theme

set_vscode_theme()


We use the ARG with two parameters as example model. I have added a `dummy` keyword arg (that does nothing) for demonstration purposes:

In [ ]:
nr_samples = 6
indexer = StateIndexer(descendants=[
    Property('loc1', max_value=nr_samples),
    Property('loc2', max_value=nr_samples)
])

initial = [0] * indexer.state_length
initial[indexer.props_to_index(loc1=1, loc2=1)] = nr_samples

def two_locus_arg_2param(state, indexer=None, dummy=None):

    transitions = []
    if state.sum() <= 1: return transitions

    for i in range(indexer.state_length):
        if state[i] == 0: continue
        pi = indexer.index_to_props(i)

        for j in range(i, indexer.state_length):
            if state[j] == 0: continue
            pj = indexer.index_to_props(j)

            same = int(i == j)
            if same and state[i] < 2:
                continue
            if not same and (state[i] < 1 or state[j] < 1):
                continue
            child = state.copy()
            child[i] -= 1
            child[j] -= 1
            loc1 = pi.descendants.loc1 + pj.descendants.loc1
            loc2 = pi.descendants.loc2 + pj.descendants.loc2
            if loc1 <= nr_samples and loc2 <= nr_samples:
                child[indexer.props_to_index(loc1=loc1, loc2=loc2)] += 1
                transitions.append([child, [state[i]*(state[j]-same)/(1+same), 0]])

        if state[i] > 0 and pi.descendants.loc1 > 0 and pi.descendants.loc2 > 0:
            child = state.copy()
            child[i] -= 1
            child[indexer.props_to_index(loc1=pi.descendants.loc1, loc2=0)] += 1
            child[indexer.props_to_index(loc1=0, loc2=pi.descendants.loc2)] += 1
            transitions.append([child, [0, 1]])

    return transitions

Start from a clean slate and enable info logging so the examples below show cache misses and hits clearly:

In [ ]:
set_log_level('INFO')
clear_caches(verbose=True)

## Graph cache

Building a graph from a callback function requires exploring the full state space, creating vertices and edges, and can take seconds to minutes for large models. The **graph cache** stores fully constructed `Graph` objects on disk so that the same model can be loaded instantly on subsequent calls.

The cache key is a SHA-256 hash of:

- The callback function's AST (abstract syntax tree), so whitespace/comment changes are ignored but code changes invalidate the cache
- All construction parameters (`ipv`, `nr_samples`, keyword arguments)

Enable the graph cache by passing `graph_cache=True` to `Graph()`. First build constructs graph from callback and saves to cache:

In [ ]:
%%time
graph = Graph(two_locus_arg_2param, ipv=initial, indexer=indexer,
    graph_cache=True)

Second build is loaded from cache:

In [ ]:
%%time
graph = Graph(two_locus_arg_2param, ipv=initial, indexer=indexer,
    graph_cache=True, dummy=42)

If you modify the callback function or pass different parameters, the cache misses and the graph is rebuilt. Even though our `dummy` keyword arg does nothing, passing a new value triggers a rebuild of the graph:

In [ ]:
%%time
graph = Graph(two_locus_arg_2param, ipv=initial, indexer=indexer,
    graph_cache=True, dummy=99)

In [ ]:
clear_caches(verbose=True)

## Parameterised reward-compute cache

When computing moments (expectation, variance, etc.) on a parameterised graph, phasic performs **Gaussian elimination** on the graph to produce a symbolic compute graph — the *parameterised reward compute graph* (PRC). The elimination is O(n³) and is the most expensive step. The output is theta-independent: the same parameterised model always produces the same PRC, regardless of the parameter values it is later evaluated with.

The PRC cache is **automatic and process-wide**. There is no opt-in keyword: when phasic builds a PRC during a `moments()` / `expectation()` / `variance()` call, it consults the cache; on a hit it loads from disk, on a miss it eliminates and writes the result for next time.

In [ ]:
# First call: cache miss, runs the eliminator.
%time graph.update_weights([2, 5]); m1 = graph.expectation()
m1

A second call within the same process is free (in-memory) — no log line, no disk I/O. But once you restart Python and rebuild the same parameterised model, the disk cache lets the new process skip the elimination:

In [ ]:
# Second call inside the same process: in-memory hit.
%time m2 = graph.expectation()
m2

Inspect the on-disk cache state:

In [ ]:
cache.param_compute_cache_info()

You can disable the cache entirely with `PHASIC_DISABLE_CACHE=1` or by `phasic.configure(cache_enabled=False)`. The cache directory is controlled by `PHASIC_CACHE_DIR` (or `phasic.configure(cache_dir=...)`), which is useful on cluster filesystems where home directories are not shared.

## Hierarchical SCC composer cache

For very large graphs the monolithic elimination dominates time and memory even when its output is cached. Phasic offers an opt-in **hierarchical SCC composer** that decomposes the graph into its strongly-connected components (SCCs), eliminates each SCC independently, and composes the per-SCC results into a parent-level answer. The wins are:

- **Parallelism.** Sibling SCCs in the same level can be eliminated concurrently (`OMP_NUM_THREADS` and `PHASIC_MAX_PARALLEL_SCCS` control this).
- **Cross-graph reuse.** Per-SCC artefacts are content-hashed, so two parents that share an SCC structurally share its cache entry.
- **Smaller cache files.** Each entry is one SCC's elimination, not the whole graph.

Enable via env var or via `configure()`:

In [ ]:
# Either:
#   export PHASIC_HIERAR_ELIMINATION=1
# Or programmatically:
configure(parallel_elimination=True)

Re-run the expectation; the composer takes the hierarchical path. Per-SCC cache files are written under the same `parameterized_reward_compute/` directory with an `scc_` prefix:

In [ ]:
cache.reset_scc_compose_stats()
graph.update_weights([2, 5])
m_hier = graph.expectation()

stats = cache.scc_compose_stats()
print(f"compose_calls:   {stats['compose_calls']}")
print(f"cache_hits:      {stats['cache_hits']}")
print(f"cache_misses:    {stats['cache_misses']}")
print(f"cache_bypassed:  {stats['cache_bypassed']}")
print(f"compose time:    {stats['total_compose_ns'] / 1e6:.2f} ms")
m_hier

`cache_hits` and `cache_misses` cover the on-disk PRC cache for each SCC. `cache_bypassed` counts SCCs whose synthetic graph fell below the size threshold (default 4 vertices) and skipped the cache entirely — small SCCs eliminate trivially and the disk I/O would dominate.

A second call on a parameter update is satisfied entirely from cache hits:

In [ ]:
cache.reset_scc_compose_stats()
graph.update_weights([3, 7])  # different theta, same structure
m_hier_2 = graph.expectation()
cache.scc_compose_stats()

### Composer controls

The composer exposes two knobs in addition to the on/off switch:

| Control | configure() field | Env var | Default | Effect |
|---|---|---|---|---|
| Min SCC size to cache | `parallel_elimination_min_subgraph` | `PHASIC_MIN_SCC_SIZE_TO_CACHE` | 4 | SCCs whose synth has fewer vertices skip the cache entirely. Set 0 to cache everything. |
| Max parallel SCCs per level | `parallel_elimination_max_concurrent` | `PHASIC_MAX_PARALLEL_SCCS` | unset (OpenMP default) | Caps SCC-level fan-out. Independent of `OMP_NUM_THREADS` — useful when you want OpenMP for each SCC's eliminator but limit the per-level memory footprint. |

`configure()` overrides existing env vars. Fields left at `None` preserve any pre-existing env vars (so SLURM job scripts can layer settings on top of `configure()` calls).

In [ ]:
configure(
    parallel_elimination=True,
    parallel_elimination_min_subgraph=8,   # only cache SCCs with synth >= 8 vertices
    parallel_elimination_max_concurrent=4,       # cap simultaneous SCC computes at 4
)

## JAX compilation cache

When running SVGD inference, JAX JIT-compiles the log-likelihood, kernel, and update functions the first time they are called. This compilation can take 1–10 seconds. The **JAX compilation cache** stores the compiled XLA code on disk so that subsequent Python sessions skip recompilation entirely.

This cache is managed by JAX itself and is enabled automatically by phasic at import time. The cache key is based on the function structure and input shapes (not values), so different parameter vectors reuse the same compiled code.

### Configuration

The default cache directory is `~/.jax_cache/`. You can change it via environment variable *before* importing JAX:

```bash
export JAX_COMPILATION_CACHE_DIR=/fast/ssd/jax_cache
```

Or programmatically with the `CompilationConfig` class:

```python
from phasic.jax_config import CompilationConfig

config = CompilationConfig.balanced()      # sensible defaults
config = CompilationConfig.max_performance()
config = CompilationConfig.fast_compile()  # for development
config.apply()
```

## Inspecting caches

Phasic provides a unified API for inspecting the cache layers.

### Overview

In [ ]:
print_all_cache_info()

### Individual cache layers

Each layer has its own inspection functions:

In [ ]:
# Graph cache.
print_graph_cache_info()

In [ ]:
# Parameterised reward-compute cache (monolithic + per-SCC entries).
cache.param_compute_cache_info()

In [ ]:
# Hierarchical SCC composer telemetry — counters of cache
# hits/misses/bypasses and cumulative compose time since the
# last reset.
cache.scc_compose_stats()

In [ ]:
# JAX compilation cache.
jax_info = cache_info()
print(f"JAX cache: {jax_info['num_files']} files, "
      f"{jax_info['total_size_mb']:.1f} MB")

## Clearing caches

| Function | What it clears |
|----------|---------------|
| `clear_caches()` | All cache layers |
| `clear_model_cache()` | Graph cache + trace cache (legacy) |
| `clear_jax_cache()` | JAX compilation cache only |
| `phasic.cache.clear_param_compute_cache()` | Parameterised reward-compute cache (monolithic + per-SCC entries) |

The parameterised reward-compute cache and the SCC composer cache share a directory; clearing one clears both.

In [ ]:
# Clear only the parameterised reward-compute cache (does not
# touch the graph cache or JAX cache).
n_removed = cache.clear_param_compute_cache()
print(f"Removed {n_removed} parameterised reward-compute entries")
cache.param_compute_cache_info()

## Caching with composed graphs

Graphs built through composition methods — `add_epoch()`, `discretize()`, `laplace_transform()`, `joint_prob_graph()` — fully support the parameterised reward-compute cache. The hash is structure-based (SHA-256 over vertices, edges, and coefficients), so the cache works identically regardless of how the graph was constructed. The same composition pipeline always produces the same hash, enabling cache hits across sessions:

```python
# Session 1: builds and caches the PRC.
graph = Graph(coalescent)
graph.update_weights([1/N0])
g1 = graph.add_epoch(t1)
g1.update_weights([1/N0, 1/N1, 1])
g2 = g1.add_epoch(t2)
g2.expectation()  # Records and caches the PRC

# Session 2: same pipeline, cache hit.
graph = Graph(coalescent)
graph.update_weights([1/N0])
g1 = graph.add_epoch(t1)
g1.update_weights([1/N0, 1/N1, 1])
g2 = g1.add_epoch(t2)
g2.expectation()  # Cache hit
```

This works because the resulting graph structure (vertices, edges, coefficient layout) is deterministic for a given composition sequence.